# Creating a Simple Agent with Tracing

In [3]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
# Test OpenAI Access
try:
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    model = os.environ.get("OPENAI_DEFAULT_MODEL", "gpt-4o-mini")
    response = client.responses.create(
        model=model,
        input="Say: We are up and running!"
    )
    print(response.output_text)
except Exception as exc:
    message = str(exc)
    if "credit_balance_exhausted" in message or "insufficient_quota" in message or "429" in message:
        print(
            "OpenAI API is configured, but your account has no remaining credits. "
            "Add billing credits in the OpenAI dashboard to continue."
        )
    else:
        print(f"OpenAI API access failed: {message}")

OpenAI API is configured, but your account has no remaining credits. Add billing credits in the OpenAI dashboard to continue.


In [4]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [5]:
with trace("Simple Nutrition Agent"):
    nutrition_agent = Agent(
        name="Nutrition Assistant",
        instructions=(
            "You are a helpful nutrition expert. Answer nutrition questions clearly "
            "and briefly, explaining the health benefits and any caveats."
        ),
    )

Let's execute the Agent:

In [6]:
with trace("Simple Nutrition Agent"):
    nutrition_agent = Agent(
        name="Nutrition Assistant",
        instructions=(
            "You are a helpful nutrition expert. Answer nutrition questions clearly "
            "and briefly, explaining the health benefits and any caveats."
        ),
    )

    try:
        result = await Runner.run(nutrition_agent, "How healthy are bananas?")
    except Exception as exc:
        message = str(exc)
        if (
            "credit_balance_exhausted" in message
            or "insufficient_quota" in message
            or "429" in message
        ):
            result = (
                "OpenAI API is configured, but your account has no remaining credits. "
                "Add billing credits in the OpenAI dashboard to continue."
            )
            print(result)
        else:
            result = f"Agent run failed: {message}"
            print(result)

print(result)

Error getting response: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}. (request_id: req_536b13f710f04c6ab125ec291a1f5119)


OpenAI API is configured, but your account has no remaining credits. Add billing credits in the OpenAI dashboard to continue.
OpenAI API is configured, but your account has no remaining credits. Add billing credits in the OpenAI dashboard to continue.


Streaming the answer to the screen, token by token

In [7]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Error streaming response: You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.


APIError: You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.

_Good Job!_